# HSSFLDON - Data Preparation Notebook

This notebook will be used to prepare a public dataset to replicate the target environment for HSSFLDON. Primarily, this means a large but unlabeled dataset for the server and small labeled datasets for each client.

In [1]:
# Make sure all directories exist
import os
os.makedirs("test", exist_ok=True)
os.makedirs("server", exist_ok=True)
os.makedirs("clients", exist_ok=True)

In [2]:
# Load dataset from Hugging Face - https://huggingface.co/datasets/ucberkeley-dlab/measuring-hate-speech
from datasets import load_dataset
dataset = load_dataset("ucberkeley-dlab/measuring-hate-speech")

/Users/chrishinkson/Programming/COMP6970 - Federated Learning/HSSFLDON/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Split off test set to be used everywhere for evaluation
import pandas as pd
df = dataset["train"].to_pandas()
df.describe() #type: ignore

,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,hatespeech,hate_speech_score,infitms,outfitms,annotator_severity,std_err,annotator_infitms,annotator_outfitms,hypothesis,annotator_age
count,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,...,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135451.000000
mean,23530.416138,5567.097812,1.281352,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,...,0.744733,-0.567428,1.034322,1.001052,-0.018817,0.300588,1.007158,1.011841,0.014589,37.910772
std,12387.194125,3230.508937,1.023542,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,...,0.932260,2.380003,0.496867,0.791943,0.487261,0.236380,0.269876,0.675863,0.613006,11.641276
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-8.340000,0.100000,0.070000,-1.820000,0.020000,0.390000,0.280000,-1.578693,18.000000
25%,18148.000000,2719.000000,0.000000,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,...,0.000000,-2.330000,0.710000,0.560000,-0.380000,0.030000,0.810000,0.670000,-0.341008,29.000000
50%,20052.000000,5602.500000,1.000000,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,...,0.000000,-0.340000,0.960000,0.830000,-0.020000,0.340000,0.970000,0.850000,0.110405,35.000000
75%,32038.250000,8363.000000,2.000000,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,...,2.000000,1.410000,1.300000,1.220000,0.350000,0.420000,1.170000,1.130000,0.449555,45.000000
max,50070.000000,11142.000000,3.000000,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,...,2.000000,6.300000,5.900000,9.000000,1.360000,1.900000,2.010000,9.000000,0.987511,81.000000


In [4]:
# Print all possible columns in original dataset
print("Columns in original dataset:")
for column in df.columns: #type: ignore
    print(f"- {column}")

Columns in original dataset:
- comment_id
- annotator_id
- platform
- sentiment
- respect
- insult
- humiliate
- status
- dehumanize
- violence
- genocide
- attack_defend
- hatespeech
- hate_speech_score
- text
- infitms
- outfitms
- annotator_severity
- std_err
- annotator_infitms
- annotator_outfitms
- hypothesis
- target_race_asian
- target_race_black
- target_race_latinx
- target_race_middle_eastern
- target_race_native_american
- target_race_pacific_islander
- target_race_white
- target_race_other
- target_race
- target_religion_atheist
- target_religion_buddhist
- target_religion_christian
- target_religion_hindu
- target_religion_jewish
- target_religion_mormon
- target_religion_muslim
- target_religion_other
- target_religion
- target_origin_immigrant
- target_origin_migrant_worker
- target_origin_specific_country
- target_origin_undocumented
- target_origin_other
- target_origin
- target_gender_men
- target_gender_non_binary
- target_gender_transgender_men
- target_gender_tran

In [5]:
# Keep only certain columns
keep = ["text", "sentiment", "respect", "insult", "humiliate", "status", "dehumanize","violence", "genocide", "attack_defend", "hatespeech", "hate_speech_score"]
trimmedDf = df[keep].copy() #type: ignore
trimmedDf.describe()

,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,attack_defend,hatespeech,hate_speech_score
count,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000
mean,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,0.678413,2.625992,0.744733,-0.567428
std,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,1.179598,1.114960,0.932260,2.380003
min,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-8.340000
25%,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,0.000000,2.000000,0.000000,-2.330000
50%,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,0.000000,3.000000,0.000000,-0.340000
75%,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,1.000000,3.000000,2.000000,1.410000
max,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.000000,6.300000


In [6]:
# Print final columns in trimmed dataset
print("Columns in trimmed dataset:")
for column in trimmedDf.columns:
    print(f"- {column}")

Columns in trimmed dataset:
- text
- sentiment
- respect
- insult
- humiliate
- status
- dehumanize
- violence
- genocide
- attack_defend
- hatespeech
- hate_speech_score


In [7]:
# Force GC for memory clearing of old dataframe
import gc
del dataset
del df
gc.collect()

45

In [8]:
# If one of the nine classification columns is above threshold, then it is considered part of target
classificationColumnThreshold = 2
classifications = ["respect", "insult", "humiliate", "status", "dehumanize","violence", "genocide", "attack_defend", "hatespeech"]

In [9]:
# Use above classification threshold to build textual target for each comment
def determineTarget(row) -> list[int]:

	# Shift all categories to binary true / false
	for column in classifications:
		if row[column] >= classificationColumnThreshold:
			row[column] = 1.0
		else:
			row[column] = 0.0

	# Create a multi-label vector for the target labels
	rowClassifications = [row[column] for column in classifications]
	return rowClassifications
	
trimmedDf["classifications"] = trimmedDf.apply(determineTarget, axis=1)
trimmedDf.describe()

,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,attack_defend,hatespeech,hate_speech_score
count,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000
mean,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,0.678413,2.625992,0.744733,-0.567428
std,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,1.179598,1.114960,0.932260,2.380003
min,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-8.340000
25%,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,0.000000,2.000000,0.000000,-2.330000
50%,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,0.000000,3.000000,0.000000,-0.340000
75%,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,1.000000,3.000000,2.000000,1.410000
max,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.000000,6.300000


In [10]:
# Prep for stratifying by binning the hate_speech_score into 10 bins
trimmedDf["hate_speech_score_bin"] = pd.qcut(trimmedDf["hate_speech_score"], q=10, labels=False)
trimmedDf.describe()

,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,attack_defend,hatespeech,hate_speech_score,hate_speech_score_bin
count,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000
mean,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,0.678413,2.625992,0.744733,-0.567428,4.490166
std,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,1.179598,1.114960,0.932260,2.380003,2.864961
min,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-8.340000,0.000000
25%,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,0.000000,2.000000,0.000000,-2.330000,2.000000
50%,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,0.000000,3.000000,0.000000,-0.340000,4.000000
75%,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,1.000000,3.000000,2.000000,1.410000,7.000000
max,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.000000,6.300000,9.000000


In [11]:
# Showcase sample before splitting
pd.set_option('display.max_colwidth', None)
print("Sample data point:")
print(trimmedDf.iloc[0])

Sample data point:
text                     Yes indeed. She sort of reminds me of the elder lady that played the part in the movie "Titanic" who was telling her story!!! And I wouldn't have wanted to cover who I really am!! I would be proud!!!! WE should be proud of our race no matter what it is!!
sentiment                                                                                                                                                                                                                                                                           0.0
respect                                                                                                                                                                                                                                                                             0.0
insult                                                                                                                                       

In [12]:
# Split into train and test sets
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(trimmedDf, test_size=0.2, random_state=2002, stratify=trimmedDf["hate_speech_score_bin"])

In [13]:
# Save test to parquet for later use
df_test.to_parquet("test/test.parquet")

In [14]:
# From the train set, we want 70% for the server and 30% for clients
df_train_server, df_train_clients = train_test_split(df_train, test_size=0.3, random_state=2002, stratify=df_train["hate_speech_score_bin"])

In [15]:
# Save server
df_train_server.to_parquet("server/server.parquet")

In [16]:
# Check distribution and size of remaining dataset for clients
df_train_clients.describe()


,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,attack_defend,hatespeech,hate_speech_score,hate_speech_score_bin
count,32534.000000,32534.000000,32534.000000,32534.000000,32534.000000,32534.000000,32534.000000,32534.000000,32534.000000,32534.000000,32534.000000,32534.000000
mean,2.955001,2.830270,2.566238,2.279062,2.695887,1.844870,1.055204,0.682302,2.624024,0.747065,-0.563835,4.490103
std,1.232783,1.309564,1.386277,1.369329,0.903670,1.400454,1.346186,1.179965,1.117385,0.933625,2.376253,2.864979
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-8.340000,0.000000
25%,2.000000,2.000000,2.000000,1.000000,2.000000,1.000000,0.000000,0.000000,2.000000,0.000000,-2.320000,2.000000
50%,3.000000,3.000000,3.000000,3.000000,3.000000,2.000000,1.000000,0.000000,3.000000,0.000000,-0.340000,4.000000
75%,4.000000,4.000000,4.000000,3.000000,3.000000,3.000000,2.000000,1.000000,3.000000,2.000000,1.410000,7.000000
max,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.000000,6.300000,9.000000


In [17]:
# Configure options for client splits
from math import floor
clientBiasStrength = 25
samplesPerClient = floor(len(df_train_clients) / 3)

In [18]:
# Map clients to specific targets for imbalance - this will be oracle network later on
clientMappings = {
    "Client_0": ["insult", "dehumanize", "violence"],
    "Client_1": ["genocide", "attack_defend", "respect"],
    "Client_2": ["sentiment", "humiliate", "hatespeech"]
}


In [19]:
# Create a copy so we don't mutate the original dataframe
availableForClientsDataframe = df_train_clients.copy()
clientDataframes = {}

In [20]:
# Split for each client
import numpy as np
for name, targets in clientMappings.items():

    # Calculate weights for the client based on data available
    is_target = availableForClientsDataframe[targets].sum(axis=1) > 0
    weights = np.where(is_target, clientBiasStrength, 1) 

    # Sample client data based on calculated weights
    client_sample = availableForClientsDataframe.sample(
        n=samplesPerClient, 
        weights=weights, 
        replace=True
    )
    
    # Store into separate dataframes
    clientDataframes[name] = client_sample
    
    # Remove from available so we don't double sample
    availableForClientsDataframe = availableForClientsDataframe.drop(client_sample.index)

print(f"Remaining 'noise' rows after splits: {len(availableForClientsDataframe)}")

Remaining 'noise' rows after splits: 7472


In [21]:
# Describe and save each client
for name, df in clientDataframes.items():
    print(f"--- {name} ---")
    print(df.describe())
    df.to_parquet(f"clients/{name}.parquet")

--- Client_0 ---
          sentiment       respect        insult     humiliate        status  \
count  10844.000000  10844.000000  10844.000000  10844.000000  10844.000000   
mean       3.231741      3.135743      2.907599      2.597473      2.835024   
std        0.977692      1.037087      1.096363      1.166170      0.887775   
min        0.000000      0.000000      0.000000      0.000000      0.000000   
25%        3.000000      2.000000      2.000000      2.000000      2.000000   
50%        4.000000      3.000000      3.000000      3.000000      3.000000   
75%        4.000000      4.000000      4.000000      4.000000      4.000000   
max        4.000000      4.000000      4.000000      4.000000      4.000000   

         dehumanize      violence      genocide  attack_defend    hatespeech  \
count  10844.000000  10844.000000  10844.000000   10844.000000  10844.000000   
mean       2.122003      1.206842      0.773515       2.833272      0.859923   
std        1.314916      1.3713